# Example 0.13.3 / 2: Exercise 16-5: Columnwise Set Covering

Companion notebook for **0.13.3 Exercises on Special Structures** in the AMPL lab Documentation, section "Exercise 16-5: Columnwise Set Covering". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 16-5: Columnwise Set Covering

Exercise 16-5 asks for a columnwise formulation of set covering: given a set $S$ and subsets
$T_1, \ldots, T_n$ with costs, choose the cheapest collection of subsets whose union is $S$. In
columnwise form the `Cover` constraints are declared first with `to_come`, and each `Use[t]`
variable then contributes its cost to the objective and a coefficient of 1 to the constraint of
every member of $T_t$ (Columnwise Formulations (AMPL lab documentation, section 10.3)). The exercise gives no costs, so this
run gives every subset cost 1 and finds the fewest subsets that cover $S = \{1,2,3,4\}$.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set S;                                   # elements to cover
set SUBSETS;                             # available subsets
set MEMBERS {SUBSETS} within S;          # elements of each subset
param cost {SUBSETS} > 0;                # cost of choosing a subset

minimize Total_Cost;                     # objective terms come with the variables
subject to Cover {s in S}: to_come >= 1; # every element covered at least once

# one column per subset: its cost in the objective, a 1 in the row of each member
var Use {t in SUBSETS} binary, obj Total_Cost cost[t], coeff {s in MEMBERS[t]} Cover[s] 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["S"] = [1, 2, 3, 4]
ampl.set["SUBSETS"] = ["T1", "T2", "T3", "T4", "T5"]
members = {"T1": [1, 2, 4], "T2": [2, 3], "T3": [1], "T4": [3, 4], "T5": [1, 3]}
for t, m in members.items():
    ampl.set["MEMBERS"][t] = m
ampl.param["cost"] = {t: 1 for t in members}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["Total_Cost"].value())
chosen = [t for t, v in ampl.var["Use"].to_dict().items() if v > 0.5]
print("chosen:", chosen)

HiGHS 1.14.0: optimal solution; objective 2
4 simplex iterations
1 branching nodes
solved 2.0
chosen: ['T1', 'T5']


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set S := 1 2 3 4;
set SUBSETS := T1 T2 T3 T4 T5;
set MEMBERS[T1] := 1 2 4;
set MEMBERS[T2] := 2 3;
set MEMBERS[T3] := 1;
set MEMBERS[T4] := 3 4;
set MEMBERS[T5] := 1 3;
param cost := T1 1  T2 1  T3 1  T4 1  T5 1;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Total_Cost, {t in SUBSETS: Use[t] > 0.5};

HiGHS 1.14.0: optimal solution; objective 2
4 simplex iterations
1 branching nodes
Total_Cost = 2

set {t in SUBSETS: Use[t] > 0.5}  := T1 T5;

